# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and accessible from the following URL. All record sets, fields, and entities are referenced by their `@id` for reproducibility and transparency.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level dataset metadata
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}\n")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")
print(f"Maintained at: {croissant_url}\n")

## 2. Data Overview
Review available record sets and their field (column) `@id`s, so you can use these IDs programmatically in later steps.

Note: All queries reference entities by their `@id`.

In [ ]:
# List available record sets and their fields
print('=== Record Sets and Fields (@id, name, dataType) ===')
recordset_ids = []
for rs in dataset.metadata.record_set:
    print(f"RecordSet @id: {rs.id} | name: {getattr(rs, 'name', '[no name]')}")
    recordset_ids.append(rs.id)
    for field in rs.field:
        field_name = getattr(field, 'name', '[no name]')
        data_type = getattr(field, 'data_type', '[no data_type]')
        print(f"    Field @id: {field.id} | name: {field_name} | dataType: {data_type}")
    print('')
if not recordset_ids:
    print('No record sets found in schema.')

## 3. Data Extraction
Load records from a designated record set into a DataFrame for further analysis.

Replace the sample variable below with the actual `@id` of a record set you wish to analyze, as printed above.

In [ ]:
# For demonstration, use the first available record set
if recordset_ids:
    main_record_set_id = recordset_ids[0]
    print(f"Using record set: {main_record_set_id}")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print("Available columns (field @id):\n", df.columns.tolist())
    display(df.head())
else:
    print("No record sets available to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply example EDA steps, including filtering and normalization for a chosen numeric field.

> Replace `<numeric_field_id>` and `<group_field_id>` below with an actual numeric field's `@id` and, optionally, a group field's `@id` from the printed columns above as necessary.

In [ ]:
# Choose a numeric field for EDA.
import numpy as np

# Manually set the field @id for demonstration.
# Example: 'cr:field:CoveragePct'
if recordset_ids and not df.empty:
    # Try to guess a numeric field. You should inspect columns printed above and use the appropriate @id.
    # Otherwise, set the name explicitly below.
    numeric_candidate_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidate_fields:
        # Try parsing some columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidate_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]

    if numeric_candidate_fields:
        numeric_field_id = numeric_candidate_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No obvious numeric field was found. Please choose the field @id manually.")
        # Example fallback
        numeric_field_id = df.columns[0]
    # Set a threshold to filter (replace as appropriate for the field)
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

    # Filter and normalize
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Optional: group by a field (e.g., by accession, type...)
    group_candidate_fields = [c for c in df.columns if c != numeric_field_id]
    if group_candidate_fields:
        group_field_id = group_candidate_fields[0]  # pick the first candidate
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric variable and, optionally, its relation to another field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if recordset_ids and not df.empty and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_candidate_fields:
        plt.figure(figsize=(10,5))
        # Show relationship for top few groups
        top_groups = df[group_field_id].value_counts().index[:5]
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} (Top 5)')
        plt.show()
else:
    print("No numeric data for visualization.")

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library to load, explore, and visualize the FAIR^2 mass spectrometry dataset via its Croissant schema. All data manipulations referenced entities by their `@id` to ensure reproducibility. For deeper analyses, further domain-specific filtering and visualization can be performed.<br>

**Tip:** Reference the dataset record set and field `@id`s (as above) for programmatic and reproducible workflows.